# LTCM Case

## 1. READING

1. Describe LTCM’s investment strategy with regard to the following aspects:

- Securities traded
- Trading frequency
- Skewness (Do they seek many small wins or a few big hits?)
- Forecasting (What is behind their selection of trades?)

__LTCM pursued relative-value arbitrage strategies aimed at exploiting pricing discrepancies across fixed-income and derivative markets. They traded sovereign bonds, interest-rate swaps, mortgage-backed securities, and equity index derivatives, typically holding positions for long horizons because convergence takes time. Their payoff profile was strongly negatively skewed: the fund sought many small, stable gains—generated through leveraged convergence trades—while being vulnerable to rare but severe losses during market disruptions. Their trade selection was grounded in quantitative valuation models that forecasted convergence of spreads, liquidity premia, and price differentials among economically similar securities.__

2. What are LTCM’s biggest advantages over its competitors?

__LTCM was able to borrow enormous amounts of money at extremely low financing rates, with very little collateral and under conditions that no other fund could obtain.__

3. The case discusses four types of funding risk facing LTCM:

- collateral haircuts
- repo maturity
- equity redemption
- loan access

The case discusses specific ways in which LTCM manages each of these risks. Briefly discuss them.

a. __Collateral haricuts:__ TTo meet the haircuts required in repo agreements, LTCM relied on the cash received through the two-way mark-to-market mechanism in their swap positions. When interest rates decreased and repo haircuts increased, LTCM would receive variation margin from the swap’s mark-to-market adjustment. This incoming cash could then be used to satisfy the higher haircut requirements.

b. __Repo Maturity:__ TLTCM used longer-term repos to reduce rollover risk associated with short-term financing. In short-term repos, rollover risk arises because banks can decide not to renew the repo from one day to the next, especially under stressed market conditions. By extending repo maturities, LTCM reduced the probability of suddenly losing access to liquidity.

c. __Equity redemption:__ To prevent investors from withdrawing capital and thereby reducing liquidity, LTCM imposed a three-year lockup period. This meant that investors were not allowed to redeem their equity during the first three years, helping the fund maintain stable capital and avoid forced liquidations.

d. __Loan access:__ Given the nature of its strategies, LTCM needed immediate access to cash to increase positions when spreads widened. To secure this liquidity, the fund arranged unsecured term loans and an unsecured line of credit, ensuring it could quickly deploy capital when convergence opportunities arose.

4. LTCM is largely in the business of selling liquidity and volatility. Describe how LTCM accounts for liquidity risk in their quantitative measurements.

__LTCM incorporated liquidity risk only in its short-term (one-month) risk models, treating liquidity-driven price movements as temporary deviations rather than structural risks. In their quantitative framework, liquidity shocks were modeled as short-lived “noise,” while long-term valuations were based solely on fundamentals. As a result, their risk measurements assumed that liquidity would always return and that spreads would eventually converge, which caused them to underestimate the true danger of a prolonged liquidity contraction.__

5. Is leverage risk currently a concern for LTCM?

__LTCM did not consider leverage risk to be a major concern because repo-financed positions were excluded from their leverage metrics. As a result, their reported leverage appeared much lower than their true economic leverage. This created a false sense of security, as the fund was actually exposed to substantial leverage risk that was not reflected in their internal measurements.__

6. Many strategies of LTCM rely on converging spreads. LTCM feels that these are almost win/win situations because of the fact that if the spread converges, they make money. If it diverges, the trade becomes even more attractive, as convergence is still expected at a future date. What is the risk in these convergence trades?

__The main risk is that in a rare market shock, the spread can move in the wrong direction and widen a lot instead of narrowing. When this happens, losses grow quickly and LTCM may not have enough cash to keep the position, forcing them to sell at a bad moment even though the trade was expected to converge later.__


## 2. Fund Performance and Attribution

### Data

- `ltcm_exhibits_data.xlsx`, `Exhibit 2`: Gross and net (total) returns of LTCM  
- `spy_data.xlsx`: SPY returns and risk-free rate (scaled tbill index)

In [2]:
from scipy.stats import norm
import statsmodels.api as sm
import pandas as pd
import math as mth
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
from scipy.stats import chi2

import sys
#sys.path.append('/Users/rodrigo/Desktop/Portfolio-and-Risk-Management/Libraries')
sys.path.append('/Users/rodrigo/Library/Mobile Documents/com~apple~CloudDocs/UChicago/2025_1_Quarter/FINM 367000 Portfolio and Risk Management/Code/Libraries')
import CAPM_decomposition as capm
import Stats_portfolio as st
import portfolio_optimization as port
import Forecasts as fc

In [25]:
source = "/Users/rodrigo/Library/Mobile Documents/com~apple~CloudDocs/UChicago/2025_1_Quarter/FINM 367000 Portfolio and Risk Management/Code/Source/ltcm_exhibits_data.xlsx"
sheet_name = "Exhibit 2"

source_2 = "/Users/rodrigo/Library/Mobile Documents/com~apple~CloudDocs/UChicago/2025_1_Quarter/FINM 367000 Portfolio and Risk Management/Code/Source/spy_data.xlsx"
sheet_name_excess_returns = "excess returns"
sheet_name_returns = "total returns"
n_temp = 12

In [52]:
#Downloading and cleaning LTCM data
df_ltcm = pd.read_excel(source, sheet_name = sheet_name)
df_ltcm = df_ltcm.iloc[3:,:]
df_ltcm.columns = ["date", "fund_capital", "gross_monthly_performance", "net_monthly_performance", "index_net_performance"]
df_ltcm.dropna(axis = 0, inplace = True)
df_ltcm["date"] = pd.to_datetime(df_ltcm["date"])
df_ltcm.set_index("date", inplace = True)
df_ltcm = df_ltcm.astype(float)
df_ltcm.head()

,fund_capital,gross_monthly_performance,net_monthly_performance,index_net_performance
date,,,,
1994-03-01,1.1,-0.011,-0.013,0.99
1994-04-01,1.1,0.014,0.008,1.00
1994-05-01,1.2,0.068,0.053,1.05
1994-06-01,1.2,-0.039,-0.029,1.02
1994-07-01,1.4,0.116,0.084,1.10


In [53]:
df_returns = pd.read_excel(source_2, sheet_name= sheet_name_returns, index_col= "date")
df_returns.index = df_returns.index.to_period('M').to_timestamp()
df_returns.dropna(axis = 0, inplace = True)
df_returns.head()

,SPY,^IRX
date,,
1994-01-01,0.034876,0.002475
1994-02-01,-0.029164,0.002800
1994-03-01,-0.047397,0.002892
1994-04-01,0.011212,0.003217
1994-05-01,0.015939,0.003475


In [54]:
df_excess_returns = pd.read_excel(source_2, sheet_name= sheet_name_excess_returns, index_col= "date")
df_excess_returns.index = df_excess_returns.index.to_period('M').to_timestamp()
df_excess_returns = pd.concat([df_excess_returns, st.get_excess_returns(df_returns["^IRX"], df_ltcm[["gross_monthly_performance", "net_monthly_performance"]])], 
                              axis = 1)
df_excess_returns.dropna(axis = 0, inplace = True)
df_excess_returns.head()

,SPY,gross_monthly_performance,net_monthly_performance
date,,,
1994-03-01,-0.050288,-0.013892,-0.015892
1994-04-01,0.007996,0.010783,0.004783
1994-05-01,0.012464,0.064525,0.049525
1994-06-01,-0.032782,-0.042450,-0.032450
1994-07-01,0.028768,0.112442,0.080442


## 1. Summary stats.

For both the gross and net series of LTCM excess returns, report the **annualized**:

- mean  
- volatility  
- Sharpe ratios  

Also report the:

- skewness  
- kurtosis  
- 5th quantile  

In [55]:
df_statistics = st.get_annualize_metrics(df_excess_returns, n_temp)
display(df_statistics[["mean", "vol", "sharpe_ratio", "skewness", "kurtosis", "Var (0.05)"]].round(2))

,mean,vol,sharpe_ratio,skewness,kurtosis,Var (0.05)
SPY,15.48,11.41,135.68,-0.41,-0.39,-4.97
gross_monthly_performance,24.36,13.62,178.81,-0.29,1.59,-3.03
net_monthly_performance,15.69,11.18,140.36,-0.81,2.93,-2.63


## 2. Compare to SPY

Comment on how these stats compare to SPY and other assets we have seen.

How much do they differ between gross and net?

__Gross returns have higher expected returns than net returns and SPY. Even though they exhibit more volatility, their Sharpe ratio is higher than the other two assets. Furthermore, their 5th quantile is lower than SPY’s.__

__On the other hand, net returns are slightly higher than SPY (by about 20 bps) and have a similar Sharpe ratio. However, SPY shows the worst 5th percentile metric in the sample.__

## 3. LFD

Estimate a linear factor decomposition of **net** LTCM excess returns on **SPY** excess returns.

Report:

- annualized alpha  
- beta  
- r-squared  

**Does LTCM deliver performance beyond SPY?**

__We obtain an alpha of around 13.51%, which indicates the presence of excess returns that are not explained by the market. Additionally, the R-squared is low (around 2%) and the beta is not significant (0.14). Therefore, LTCM’s performance is not explained by the market.__

In [58]:
list_names_ex3 = ["net_monthly_performance"]
list_factors_ex3 = [["SPY"]]

df_info_ex3, df_predict_ex_3 = fc.get_annualized_regression(df_excess_returns[["net_monthly_performance"]], df_excess_returns[["SPY"]], 
                                                            list_names_ex3, list_factors_ex3, n_temp)
display(df_info_ex3.round(4))

net_monthly_performance                             \
                                          Alpha Beta_SPY r_squared   std_e   
net_monthly_performance                  0.1351   0.1409    0.0207  0.1106   

                                                           
                        pi_val_SPY Treynor_SPY Info_ratio  
net_monthly_performance     0.3043      1.1135     1.2212

## 4. Nonlinear Exposure

Let's check for non-linear market exposure. Run the following regression on LTCM’s **net excess returns**:

$$
\tilde{r}^{\text{ltcm}}_t = \alpha + \beta_{\text{linear}}\tilde{r}^m_t + \beta_{\text{quad}}(\tilde{r}^m_t)^2 + \epsilon_t
$$

Report:

- annualized alpha  
- the linear and quadratic betas  
- r-squared  

In [66]:
df_factors_ex4 = df_excess_returns[["SPY"]].copy()
df_factors_ex4["SPY_squared"] = df_excess_returns["SPY"] * df_excess_returns["SPY"]

list_names_ex4 = ["Exercise_3","Exercise_4"]
list_factors_ex4 = [["SPY"],["SPY", "SPY_squared"]]

df_info_ex4, df_predict_ex_4 = fc.get_annualized_regression(df_excess_returns[["net_monthly_performance"]], df_factors_ex4, 
                                                            list_names_ex4, list_factors_ex4, n_temp)
display(df_info_ex4["Exercise_4"].round(4))

,Alpha,Beta_SPY,Beta_SPY_squared,r_squared,std_e,pi_val_SPY,pi_val_SPY_squared,Treynor_SPY,Treynor_SPY_squared,Info_ratio
net_monthly_performance,0.1626,0.1687,-2.1583,0.0278,0.1102,0.2474,0.5477,0.9297,-0.0727,1.4757


## 5. Questions

- Does the quadratic market factor do much to increase the overall LTCM variation explained by the market?

__The quadratic term increases the explained variation only marginally (about 70 bps), which indicates that the nonlinear market exposure does not significantly improve the model’s explanatory power.__

- From the regression evidence, does LTCM’s market exposure behave as if it is long market options or short market options?

__From the regression results, LTCM behaves as if it is short market options. The negative quadratic beta indicates negative convexity (short gamma), meaning LTCM tends to lose money when market volatility increases.__

- Should we describe LTCM as being positively or negatively exposed to market volatility?

__Since the SPY-squared beta is negative, we can conclude that LTCM had negative exposure to market volatility. This is consistent with being short market convexity (or short gamma). In practice, this means that LTCM tended to lose money when market volatility increased, regardless of whether the market moved up or down.__


## 6.

Let’s try to pinpoint the nature of LTCM’s nonlinear exposure.  
Does it come more from exposure to up-markets or down-markets?  
Run the following regression on LTCM’s **net excess returns**:

$$
\tilde{r}^{ltcm}_t
=
\alpha 
+ \beta\, \tilde{r}^m_t
+ \beta_u \max(\tilde{r}^m_t - k_1,\; 0)
+ \beta_d \max(k_2 - \tilde{r}^m_t,\; 0)
+ \epsilon_t
$$

where  
- $k_1 = 0.03$  
- $k_2 = -0.03$

**Report:**

- annualized alpha  
- market beta, the **up** and **down** betas  
- r-squared  

In [65]:
k1 = 0.03
k2 = -0.03

df_factors_ex5 = df_excess_returns[["SPY"]].copy()
df_factors_ex5["up_spy"] = (df_factors_ex5["SPY"] - k1).apply(lambda x: max(x, 0))
df_factors_ex5["down_spy"] = (k2 - df_factors_ex5["SPY"]).apply(lambda x: max(x, 0))

list_names_ex5 = ["Exercise_5"]
list_factors_ex5 = [["SPY", "up_spy", "down_spy"]]

df_info_ex5, df_predict_ex_5 = fc.get_annualized_regression(df_excess_returns[["net_monthly_performance"]], df_factors_ex5, 
                                                            list_names_ex5, list_factors_ex5, n_temp)
display(df_info_ex5.round(4))

Exercise_5                                     \
                             Alpha Beta_SPY Beta_up_spy Beta_down_spy   
net_monthly_performance     0.1094   0.4345     -0.7219        1.0423   

                                                                   \
                        r_squared  std_e pi_val_SPY pi_val_up_spy   
net_monthly_performance    0.0486  0.109     0.1327        0.2617   

                                                                    \
                        pi_val_down_spy Treynor_SPY Treynor_up_spy   
net_monthly_performance          0.3673      0.3611        -0.2173   

                                                     
                        Treynor_down_spy Info_ratio  
net_monthly_performance           0.1505     1.0038

## 7. Questions

- Is LTCM long or short the call-like factor? And the put-like factor?

__LTCM is short the call factor and long the put factor__

- Which factor moves LTCM more, the call-like factor, or the put-like factor?

__Taking the absolute values of both betas, the put-like factor has the stronger impact on LTCM’s returns.__

- In the previous problem, you commented on whether LTCM is positively or negatively exposed to market volatility. Using this current regression, does this volatility exposure come more from being long the market’s upside? Short the market’s downside? Something else?

__The regression results suggest that LTCM tends to generate positive returns during market downturns while incurring losses in rising markets. Nevertheless, as established in Exercise 4, LTCM exhibits losses under episodes of heightened volatility irrespective of market direction, reflecting its structural vulnerability to extreme volatility shocks.__